In [1]:
from langchain_community.retrievers import WikipediaRetriever

In [2]:
retriever = WikipediaRetriever(top_k_results=2, lang='en')

In [3]:
query = "who is prithvi narayan shah?"

docs = retriever.invoke(query)

In [4]:
docs

[Document(metadata={'title': 'Prithvi Narayan Shah', 'summary': "Prithvi Narayan Shah was the last king of the Gorkha Kingdom and the first monarch of the unified Kingdom of Nepal, also referred to as the Kingdom of Gorkha. He is credited with initiating the unification of Nepal and relocating the royal seat of power to Kathmandu. His reign marked the end of the medieval period and the beginning of the modern history of Nepal.\nPrithvi Narayan Shah played a central role in the formation of modern Nepal through his campaigns of territorial consolidation. He is considered a prominent figure in Nepali history, and his efforts at political unification and national identity continue to be recognized in contemporary narratives.\nScholars have drawn comparisons between his historical role and that of George Washington in the United States, citing his vision and leadership within the context of 18th-century South Asia. Some interpretations highlight his efforts to mobilize public support and e

In [5]:
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")


--- Result 1 ---
Content:
Prithvi Narayan Shah was the last king of the Gorkha Kingdom and the first monarch of the unified Kingdom of Nepal, also referred to as the Kingdom of Gorkha. He is credited with initiating the unification of Nepal and relocating the royal seat of power to Kathmandu. His reign marked the end of the medieval period and the beginning of the modern history of Nepal.
Prithvi Narayan Shah played a central role in the formation of modern Nepal through his campaigns of territorial consolidation. He is considered a prominent figure in Nepali history, and his efforts at political unification and national identity continue to be recognized in contemporary narratives.
Scholars have drawn comparisons between his historical role and that of George Washington in the United States, citing his vision and leadership within the context of 18th-century South Asia. Some interpretations highlight his efforts to mobilize public support and establish a foundation for a unified stat

vector store retriever

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint, ChatHuggingFace
from langchain_classic.vectorstores import Chroma
from langchain_classic.schema import Document
from dotenv import load_dotenv
import os 
load_dotenv()

f:\SAMARPAN\LangChain\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [29]:
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation",
    huggingfacehub_api_token= os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
)

model = ChatHuggingFace(llm=llm)
embedding_model = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7114.61it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [9]:
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"
)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [11]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [12]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


MMR

In [13]:
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [15]:
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="my_collection2"
)

In [16]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2,"lambda_mult":0.5} #lambda_mult = relevance-diversity balance range=[0,1] is value is set 1 then mmr acts as similar search. if 0 then diversified answer.
    )

In [17]:
query = "What is langchain?"
results = retriever.invoke(query)

In [18]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain supports Chroma, FAISS, Pinecone, and more.

--- Result 2 ---
LangChain is used to build LLM based applications.


Multi query retriever

In [20]:
from langchain_classic.retrievers import MultiQueryRetriever

In [21]:
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [23]:
vectorstore = Chroma.from_documents(
    documents=all_docs,
    embedding=embedding_model,
    collection_name="my_collection3"
)

In [24]:
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [34]:
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation",
    huggingfacehub_api_token= os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
)
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=model
)

In [26]:
query = "How to improve energy levels and maintain balance?"

In [35]:
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [36]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 5 ---
Photosynthesis enables plants to produce energy by converting sunlight.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 4 ---
Consuming leafy greens and fruits helps detox the body and improve l

ContextualCompressionRetriever

In [37]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

In [38]:
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [40]:
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="my_collection4"
)

In [41]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [45]:
# Set up the compressor using an LLM
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
compressor = LLMChainExtractor.from_llm(model)

In [46]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [52]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis enables plants to produce energy by converting sunlight.

--- Result 2 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 3 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
